# 第7部 発展

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/07_advanced.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 7-1 会場での調整：少ないデータで追加学習

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part7-advanced/calibration/

### 手を動かす 1：家のモデルと「会場」のデータ

In [ ]:
import numpy as np
import pandas as pd
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, keras_loss, build_ballnet, evaluate
from rcjml.sim import make_dataset, Session

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train, test = split_by_session(df)
last = train.session.max()
home_fit, home_val = train[train.session < last], train[train.session == last]

def fit(model, trn, val, epochs=400, lr=1e-3, patience=20):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=keras_loss())
    model.fit(features(trn, cfg), targets(trn), validation_data=(features(val, cfg), targets(val)),
              epochs=epochs, batch_size=64, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=patience, restore_best_weights=True)])
    return model

keras.utils.set_random_seed(0)
home = fit(build_ballnet(), home_fit, home_val)

# 会場：左前（70°）の方向から強い光が入り、ボールが暗め
venue_condition = Session(ambient=1.5, ambient_dir_deg=70.0, ambient_dir=5.0, ball_scale=0.75)
venue = make_dataset(cfg, n=3000, sessions=6, seed=77, venue=venue_condition)
venue_cal = venue[venue.session == 0]      # 会場で最初に集めるデータ（ここから少しだけ使う）
venue_test = venue[venue.session >= 1]     # その後の試合のつもり

mae = lambda m, d: evaluate(m.predict(features(d, cfg), verbose=0), d)["angle_MAE"]
print(f"家のモデル：家で {mae(home, test):.1f}°、会場で {mae(home, venue_test):.1f}°")

### 手を動かす 2：3 つの方法で調整する

In [ ]:
def calibrate(n):
    cal = venue_cal.sample(n, random_state=0)
    c_fit, c_val = cal.iloc[: int(n * 0.8)], cal.iloc[int(n * 0.8):]

    keras.utils.set_random_seed(0)                    # (1) ゼロから
    scratch = fit(build_ballnet(), c_fit, c_val)

    keras.utils.set_random_seed(0)                    # (2) 家の重みから追加学習
    ft = build_ballnet()
    ft.set_weights(home.get_weights())
    ft = fit(ft, c_fit, c_val, epochs=200, lr=3e-4, patience=10)

    keras.utils.set_random_seed(0)                    # (3) 家のデータも混ぜて追加学習
    mix = build_ballnet()
    mix.set_weights(home.get_weights())
    mixed = pd.concat([home_fit.sample(2000, random_state=0), pd.concat([c_fit] * 5)])
    mix = fit(mix, mixed, c_val, epochs=200, lr=3e-4, patience=10)
    return scratch, ft, mix

for n in [50, 200, 500]:
    scratch, ft, mix = calibrate(n)
    print(f"会場のデータ {n:3d} 個: ゼロから {mae(scratch, venue_test):.1f}°  "
          f"追加学習 {mae(ft, venue_test):.1f}°（家で {mae(ft, test):.1f}°）  "
          f"混ぜて追加学習 {mae(mix, venue_test):.1f}°（家で {mae(mix, test):.1f}°）")

## 7-2 ボールがないとき：自信のないときは「わからない」と言う

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part7-advanced/no-ball/

### 手を動かす 1：今のモデルに「ボールなし」を見せる

In [ ]:
import numpy as np
import pandas as pd
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, keras_loss, build_ballnet, decode, evaluate
from rcjml.sim import make_no_ball

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train, test = split_by_session(df)
last = train.session.max()
fit, val = train[train.session < last], train[train.session == last]

def train_ballnet(trn, val, make_targets=targets, seed=0):
    keras.utils.set_random_seed(seed)
    m = build_ballnet()
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
    m.fit(features(trn, cfg), make_targets(trn), validation_data=(features(val, cfg), make_targets(val)),
          epochs=400, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])
    return m

home = train_ballnet(fit, val)

no_ball_train = make_no_ball(cfg, 1500, sessions=9, seed=1)   # 外乱光・ほかの赤外線だけ
no_ball_test = make_no_ball(cfg, 500, sessions=3, seed=2)
conf = lambda m, d: decode(m.predict(features(d, cfg), verbose=0))["confidence"]

c_ball, c_none = conf(home, test), conf(home, no_ball_test)
print(f"自信の平均  ボールあり {c_ball.mean():.2f}  ボールなし {c_none.mean():.2f}")

### 手を動かす 2：「ボールなし」を混ぜて学習する

In [ ]:
def targets_with_no_ball(d):
    T = targets(d.fillna({"angle_deg": 0.0, "distance_cm": 300.0}))
    none = d["visible"].eq(0).values if "visible" in d else np.zeros(len(d), bool)
    T[none, 0] = 0.0      # cos
    T[none, 1] = 0.0      # sin  → 長さ 0 ＝「わからない」
    return T

trn = pd.concat([fit, no_ball_train[no_ball_train.session < 8]])
vl = pd.concat([val, no_ball_train[no_ball_train.session == 8]])
aware = train_ballnet(trn, vl, make_targets=targets_with_no_ball)

c_ball2, c_none2 = conf(aware, test), conf(aware, no_ball_test)
for th in [0.3, 0.5, 0.7]:
    print(f"しきい値 {th}:  ボールありを見のがす {np.mean(c_ball2 < th):.1%}"
          f"  ボールなしを『ある』と言う {np.mean(c_none2 >= th):.1%}")
print("角度の平均のずれ:", round(evaluate(aware.predict(features(test, cfg), verbose=0), test)["angle_MAE"], 2), "°")

### 手を動かす 3：どんなボールで自信が低くなるか

In [ ]:
low = c_ball2 < 0.5
print("自信が低いボールの距離（中央値）:", np.median(test.distance_cm.values[low]), "cm")
print("全部のボールの距離（中央値）:", np.median(test.distance_cm.values), "cm")
print("自信が低いボールのうち、さえぎられていた割合:", round(test.occluded.values[low].mean(), 2))

## 7-3 模倣学習：お手本の動きをまねる

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part7-advanced/imitation/

### 手を動かす 1：入力と答えを決める

In [ ]:
import numpy as np
import keras
from rcjml.sim.field import chase_rule, rollout, success_rate, MAX_SPEED
from rcjml.export import layers_from_keras, forward

def expert(o):                      # お手本（5-5 の回り込み）
    return chase_rule(o["angle_deg"], o["distance_cm"], o["holding"], o["goal_angle_deg"])

def feat(o):                        # 入力：角度は cos と sin に（第1部）
    a, g = np.deg2rad(o["angle_deg"]), np.deg2rad(o["goal_angle_deg"])
    return [np.cos(a), np.sin(a), np.log(max(o["distance_cm"], 1)) / 5,
            float(o["holding"]), np.cos(g), np.sin(g)]

def targ(action):                   # 答え：進む向き（cos, sin）と速さ（0〜1）
    direction, speed = action
    t = np.deg2rad(direction)
    return [np.cos(t), np.sin(t), speed / MAX_SPEED]

print(success_rate(expert, n=200, seed=100))    # お手本の成績

### 手を動かす 2：お手本を記録して、まねを学習する

In [ ]:
def collect(policy, n_episodes, seed, teacher=None):
    rng = np.random.default_rng(seed)
    X, Y = [], []
    for _ in range(n_episodes):
        _, observations, actions = rollout(policy, rng, teacher=teacher)
        X += [feat(o) for o in observations]
        Y += [targ(a) for a in actions]
    return np.array(X, np.float32), np.array(Y, np.float32)

def train(X, Y, seed=0):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.Input((6,)),
                          keras.layers.Dense(64, activation="relu"),
                          keras.layers.Dense(64, activation="relu"),
                          keras.layers.Dense(3)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    m.fit(X, Y, epochs=60, batch_size=256, verbose=0, validation_split=0.1,
          callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
    return layers_from_keras(m)      # 速く動かすため、第4部の NumPy の推論を使う

def as_policy(layers):
    def policy(o):
        y = forward(layers, np.array([feat(o)], np.float32))[0]
        return float(np.rad2deg(np.arctan2(y[1], y[0]))), float(np.clip(y[2], 0, 1) * MAX_SPEED)
    return policy

X, Y = collect(expert, 50, seed=1)
student = as_policy(train(X, Y))
print("お手本 50 本でまねしたロボット:", success_rate(student, n=200, seed=100))

### 手を動かす 3：お手本の数と DAgger

In [ ]:
for seed in [0, 1, 2]:
    bc, dagger = [], []
    for n in [10, 20, 30, 40]:                                      # お手本だけ
        X, Y = collect(expert, n, seed=1 + seed * 100)
        bc.append(success_rate(as_policy(train(X, Y, seed)), n=200, seed=100)["goal_rate"])
    X, Y = collect(expert, 10, seed=1 + seed * 100)                  # DAgger：お手本 10 本から
    layers = train(X, Y, seed)
    dagger.append(success_rate(as_policy(layers), n=200, seed=100)["goal_rate"])
    for it in range(1, 4):                                          # 10 本ずつ足す
        Xn, Yn = collect(as_policy(layers), 10, seed=10 + it + seed * 100, teacher=expert)
        X, Y = np.vstack([X, Xn]), np.vstack([Y, Yn])
        layers = train(X, Y, seed)
        dagger.append(success_rate(as_policy(layers), n=200, seed=100)["goal_rate"])
    print(f"seed {seed}: お手本だけ {bc}  DAgger {dagger}")

### 手を動かす 4：お手本が下手だったら

In [ ]:
rng_noise = np.random.default_rng(5)
def noisy_expert(o):
    d, s = expert(o)
    return d + rng_noise.normal(0, 20), s * rng_noise.uniform(0.7, 1.0)

X, Y = collect(noisy_expert, 200, seed=1)
student = as_policy(train(X, Y))
print("ぶれたお手本:", success_rate(noisy_expert, n=200, seed=100))
print("それをまねたロボット:", success_rate(student, n=200, seed=100))

## 7-4 追跡と自己位置：時間と場所を使う

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part7-advanced/tracking/

### 手を動かす 1：α-β フィルタを作る

In [ ]:
import numpy as np
from rcjml.sim.field import chase_rule, rel_ball, goal_angle, new_episode, step

def wrap(a):
    return (a + 180) % 360 - 180

class AlphaBeta:
    def __init__(self, alpha=0.5, beta=0.1, gate=40.0, dt=0.05, max_miss=3):
        self.alpha, self.beta, self.gate, self.dt, self.max_miss = alpha, beta, gate, dt, max_miss
        self.x, self.v, self.miss = None, 0.0, 0

    def update(self, z):
        """z は測った角度（来なかったら None）。追跡した角度を返す。"""
        if self.x is None:                  # まだ一度も見ていない
            if z is not None:
                self.x = z
            return self.x
        pred = wrap(self.x + self.v * self.dt)
        if z is not None and abs(wrap(z - pred)) < self.gate:
            r = wrap(z - pred)
            self.x = wrap(pred + self.alpha * r)
            self.v += self.beta * r / self.dt
            self.miss = 0
        else:                               # 来なかった、またはゲートの外
            self.x, self.v = pred, self.v * 0.9
            self.miss += 1
            if z is not None and self.miss > self.max_miss:
                self.x, self.v, self.miss = z, 0.0, 0     # リセット
        return self.x

### 手を動かす 2：にせ物の混じった測定で比べる

In [ ]:
def run(make_filter=None, n=200, seed=0, sigma=8, p_out=0.08, p_drop=0.15):
    rng = np.random.default_rng(seed)
    errors = []
    for _ in range(n):
        ep = new_episode(rng)
        f = make_filter() if make_filter else None
        last = None
        for t in range(400):
            a, d = rel_ball(ep.robot, ep.ball)
            if rng.random() < p_drop:
                z = None
            else:
                z = a + rng.normal(0, sigma)
                if rng.random() < p_out:
                    z += rng.choice([-1, 1]) * rng.uniform(60, 150)
                z = wrap(z)
            if f:
                est = f.update(z)
            else:                          # フィルタなし：来なかったら前の値
                est = z if z is not None else last
                last = est
            if est is None:
                est = 0.0
            if not ep.holding:
                errors.append(abs(wrap(est - a)))
            step(ep, *chase_rule(a, d, ep.holding, goal_angle(ep.robot)))
            if ep.goal or ep.out:
                break
    e = np.array(errors)
    return e.mean(), np.percentile(e, 95)

for name, mk in [("フィルタなし", None),
                 ("α-β（ゲートなし）", lambda: AlphaBeta(gate=999)),
                 ("α-β（ゲート＋リセット）", lambda: AlphaBeta()),
                 ("α-β α=0.3", lambda: AlphaBeta(alpha=0.3, beta=0.05)),
                 ("α-β α=0.7", lambda: AlphaBeta(alpha=0.7, beta=0.2))]:
    mae, p95 = run(mk)
    print(f"{name:18s} 平均のずれ {mae:5.1f}°   悪いほうから 5% {p95:5.1f}°")

### 手を動かす 3：柱の角度から位置を出す

In [ ]:
from rcjml.sim.field import FIELD_H, GOAL_W

POSTS = np.array([[-GOAL_W / 2, FIELD_H / 2], [GOAL_W / 2, FIELD_H / 2]])   # 相手ゴールの左右の柱

def unit(theta_deg):
    t = np.deg2rad(theta_deg)
    return np.array([-np.sin(t), np.cos(t)])       # 正面 0°、左 +（1-4 の決まり）

def locate(a_left, a_right):
    """左右の柱が見える角度から、ロボットの位置 (x, y) を求める。"""
    u1, u2 = unit(a_left), unit(a_right)
    # 柱0 - t1・u1 = 柱1 - t2・u2 を t1, t2 について解く
    t1, t2 = np.linalg.solve(np.column_stack([-u1, u2]), POSTS[1] - POSTS[0])
    return POSTS[0] - t1 * u1

robot = np.array([20.0, -30.0])
print(locate(rel_ball(robot, POSTS[0])[0], rel_ball(robot, POSTS[1])[0]))   # → [ 20. -30.]

rng = np.random.default_rng(0)
err, far = [], []
for _ in range(2000):
    r = np.array([rng.uniform(-80, 80), rng.uniform(-110, 90)])
    al = rel_ball(r, POSTS[0])[0] + rng.normal(0, 1.5)      # カメラのずれ ±1.5°
    ar = rel_ball(r, POSTS[1])[0] + rng.normal(0, 1.5)
    err.append(np.linalg.norm(locate(al, ar) - r))
    far.append(FIELD_H / 2 - r[1])
err, far = np.array(err), np.array(far)
print(f"位置のずれ（中央値）: 全体 {np.median(err):.1f} cm、"
      f"ゴールに近い（80 cm 未満）{np.median(err[far < 80]):.1f} cm、遠い（160 cm 以上）{np.median(err[far > 160]):.1f} cm")